In [2]:
import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from lightgbm import LGBMClassifier
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, OneHotEncoder


RANDOM_STATE = 42
DATA_PATH = Path("/Users/apple/Documents/cfa/IT5006/models/final_crime_dataset_with_temporal.csv")
OUTPUT_DIR = Path("/Users/apple/Documents/cfa/IT5006/models")

TARGET_COL = "crime_category"
BASE_FEATURES = [
    "hour",
    "day_of_week",
    "month",
    "is_weekend",
    "community_area_id",
    "distance_to_nearest_station",
    "stations_within_500m",
    "community_type",
]

NUMERIC_FEATURES = [
    "hour",
    "day_of_week",
    "month",
    "is_weekend",
    "community_area_id",
    "distance_to_nearest_station",
    "stations_within_500m",
]

CATEGORICAL_FEATURES = ["community_type"]


def load_dataset(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(
            f"Dataset not found at {path.resolve()}. Place final_crime_dataset_with_temporal.csv next to this script or update DATA_PATH."
        )

    df = pd.read_csv(path)

    required_cols = [TARGET_COL] + BASE_FEATURES
    missing = [col for col in required_cols if col not in df.columns]
    if missing:
        raise ValueError(f"Dataset is missing required columns: {missing}")

    df = df[required_cols].dropna().copy()

    # Ensure stable dtypes for inference
    int_like_cols = ["hour", "day_of_week", "month", "is_weekend", "community_area_id", "stations_within_500m"]
    float_like_cols = ["distance_to_nearest_station"]

    for col in int_like_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    for col in float_like_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    df["community_type"] = df["community_type"].astype(str).str.strip()
    df = df.dropna().copy()

    for col in int_like_cols:
        df[col] = df[col].astype(int)
    for col in float_like_cols:
        df[col] = df[col].astype(float)

    return df


def build_pipeline() -> Pipeline:
    preprocessor = ColumnTransformer(
        transformers=[
            (
                "cat",
                OneHotEncoder(handle_unknown="ignore", sparse_output=False),
                CATEGORICAL_FEATURES,
            ),
            ("num", "passthrough", NUMERIC_FEATURES),
        ],
        remainder="drop",
    )

    model = LGBMClassifier(
        objective="multiclass",
        class_weight="balanced",
        n_estimators=200,
        learning_rate=0.08,
        num_leaves=31,
        subsample=0.9,
        colsample_bytree=0.9,
        random_state=RANDOM_STATE,
        verbose=-1,
    )

    return Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", model),
    ])


def compute_feature_stats(df: pd.DataFrame) -> dict:
    stats = {}
    for col in NUMERIC_FEATURES:
        stats[col] = {
            "mean": float(df[col].mean()),
            "std": float(df[col].std(ddof=0)) if float(df[col].std(ddof=0)) > 0 else 0.0,
            "min": float(df[col].min()),
            "max": float(df[col].max()),
        }
    return stats


def top_k_preview(prob_row: np.ndarray, class_names: list[str], k: int = 3) -> list[dict]:
    idx = np.argsort(prob_row)[::-1][:k]
    return [
        {"label": class_names[i], "probability": float(prob_row[i])}
        for i in idx
    ]


def main() -> None:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    df = load_dataset(DATA_PATH)
    X = df[BASE_FEATURES].copy()
    y = df[TARGET_COL].copy()

    label_encoder = LabelEncoder()
    y_encoded = label_encoder.fit_transform(y)

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y_encoded,
        test_size=0.2,
        random_state=RANDOM_STATE,
        stratify=y_encoded,
    )

    pipeline = build_pipeline()
    pipeline.fit(X_train, y_train)

    y_pred = pipeline.predict(X_test)
    y_prob = pipeline.predict_proba(X_test)

    accuracy = accuracy_score(y_test, y_pred)
    report = classification_report(
        y_test,
        y_pred,
        target_names=label_encoder.classes_,
        output_dict=True,
        zero_division=0,
    )

    pipeline_path = OUTPUT_DIR / "chicago_crime_pipeline.pkl"
    labels_path = OUTPUT_DIR / "label_classes.json"
    metadata_path = OUTPUT_DIR / "model_metadata.json"
    feature_stats_path = OUTPUT_DIR / "feature_stats.json"
    sample_payloads_path = OUTPUT_DIR / "sample_payloads.json"

    joblib.dump(pipeline, pipeline_path)

    with open(labels_path, "w", encoding="utf-8") as f:
        json.dump(list(label_encoder.classes_), f, indent=2, ensure_ascii=False)

    metadata = {
        "model_type": "LightGBM multiclass pipeline",
        "target_column": TARGET_COL,
        "feature_columns": BASE_FEATURES,
        "numeric_features": NUMERIC_FEATURES,
        "categorical_features": CATEGORICAL_FEATURES,
        "random_state": RANDOM_STATE,
        "n_classes": int(len(label_encoder.classes_)),
        "class_labels": list(label_encoder.classes_),
        "metrics": {
            "accuracy": float(accuracy),
            "macro_f1": float(report.get("macro avg", {}).get("f1-score", 0.0)),
            "weighted_f1": float(report.get("weighted avg", {}).get("f1-score", 0.0)),
        },
        "notes": [
            "This deployment artifact is reconstructed from the provided notebooks.",
            "It uses the stable base feature set for inference.",
            "Any more complex engineered features from exploratory notebook cells are intentionally excluded for deployment simplicity.",
        ],
    }

    with open(metadata_path, "w", encoding="utf-8") as f:
        json.dump(metadata, f, indent=2, ensure_ascii=False)

    with open(feature_stats_path, "w", encoding="utf-8") as f:
        json.dump(compute_feature_stats(df), f, indent=2, ensure_ascii=False)

    sample_payloads = []
    sample_rows = X_test.head(3).copy()
    sample_probs = pipeline.predict_proba(sample_rows)
    for i, (_, row) in enumerate(sample_rows.iterrows()):
        payload = row.to_dict()
        payload["top_3_predictions"] = top_k_preview(sample_probs[i], list(label_encoder.classes_), k=3)
        sample_payloads.append(payload)

    with open(sample_payloads_path, "w", encoding="utf-8") as f:
        json.dump(sample_payloads, f, indent=2, ensure_ascii=False)

    print("Saved:")
    print(f"  - {pipeline_path}")
    print(f"  - {labels_path}")
    print(f"  - {metadata_path}")
    print(f"  - {feature_stats_path}")
    print(f"  - {sample_payloads_path}")
    print()
    print(f"Validation accuracy: {accuracy:.4f}")
    print("Top-3 preview from first test sample:")
    print(json.dumps(sample_payloads[0]["top_3_predictions"], indent=2, ensure_ascii=False))


if __name__ == "__main__":
    main()


/Users/apple/.pyenv/versions/3.11.9/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/apple/.pyenv/versions/3.11.9/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Saved:
  - /Users/apple/Documents/cfa/IT5006/models/chicago_crime_pipeline.pkl
  - /Users/apple/Documents/cfa/IT5006/models/label_classes.json
  - /Users/apple/Documents/cfa/IT5006/models/model_metadata.json
  - /Users/apple/Documents/cfa/IT5006/models/feature_stats.json
  - /Users/apple/Documents/cfa/IT5006/models/sample_payloads.json

Validation accuracy: 0.2481
Top-3 preview from first test sample:
[
  {
    "label": "Violent",
    "probability": 0.20891782954090485
  },
  {
    "label": "Other",
    "probability": 0.18830622316085385
  },
  {
    "label": "Property",
    "probability": 0.1804040282386866
  }
]


/Users/apple/.pyenv/versions/3.11.9/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [8]:
pip install fastapi uvicorn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.4/117.4 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.8/68.8 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.3/108.3 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 463.6/463.6 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 15.9 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.7/72.7 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.4/114.4 kB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.0/71.0 kB 8.4 MB/s eta 0:00:00

[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [9]:
import json
from pathlib import Path
from typing import List

import joblib
import pandas as pd
from fastapi import FastAPI
from pydantic import BaseModel, Field, field_validator


MODEL_DIR = Path("/Users/apple/Documents/cfa/IT5006/models")
PIPELINE_PATH = MODEL_DIR / "chicago_crime_pipeline.pkl"
LABELS_PATH = MODEL_DIR / "label_classes.json"
METADATA_PATH = MODEL_DIR / "model_metadata.json"


DAY_NAME_TO_INT = {
    "Monday": 0,
    "Tuesday": 1,
    "Wednesday": 2,
    "Thursday": 3,
    "Friday": 4,
    "Saturday": 5,
    "Sunday": 6,
}


class PredictionInput(BaseModel):
    hour: int = Field(..., ge=0, le=23)
    day_of_week: int | str = Field(..., description="0-6 or Monday-Sunday")
    month: int = Field(..., ge=1, le=12)
    is_weekend: int | bool = Field(..., description="0/1 or boolean")
    community_area_id: int = Field(..., ge=1, le=77)
    distance_to_nearest_station: float = Field(..., ge=0)
    stations_within_500m: int = Field(..., ge=0)
    community_type: str = Field(..., min_length=1)

    @field_validator("day_of_week", mode="before")
    @classmethod
    def normalize_day_of_week(cls, value):
        if isinstance(value, str):
            value = value.strip()
            if value.isdigit():
                value = int(value)
            else:
                if value not in DAY_NAME_TO_INT:
                    raise ValueError("day_of_week must be 0-6 or a valid weekday name")
                value = DAY_NAME_TO_INT[value]
        if not isinstance(value, int) or not (0 <= value <= 6):
            raise ValueError("day_of_week must be an integer from 0 to 6")
        return value

    @field_validator("is_weekend", mode="before")
    @classmethod
    def normalize_is_weekend(cls, value):
        if isinstance(value, bool):
            return int(value)
        if isinstance(value, str):
            value = value.strip().lower()
            if value in {"yes", "true", "1"}:
                return 1
            if value in {"no", "false", "0"}:
                return 0
        if isinstance(value, int) and value in {0, 1}:
            return value
        raise ValueError("is_weekend must be 0/1, true/false, or yes/no")

    @field_validator("community_type")
    @classmethod
    def normalize_community_type(cls, value: str):
        value = value.strip()
        if not value:
            raise ValueError("community_type cannot be empty")
        return value


class TopPrediction(BaseModel):
    label: str
    probability: float


class PredictionResponse(BaseModel):
    predicted_label: str
    predicted_probability: float
    top_3_predictions: List[TopPrediction]
    model_version: str


app = FastAPI(title="Chicago Crime Prediction API")

pipeline = None
class_labels = None
metadata = None
feature_columns = None


@app.on_event("startup")
def load_assets() -> None:
    global pipeline, class_labels, metadata, feature_columns

    pipeline = joblib.load(PIPELINE_PATH)

    with open(LABELS_PATH, "r", encoding="utf-8") as f:
        class_labels = json.load(f)

    with open(METADATA_PATH, "r", encoding="utf-8") as f:
        metadata = json.load(f)

    feature_columns = metadata["feature_columns"]


@app.get("/")
def root():
    return {"status": "ok", "service": "Chicago Crime Prediction API"}


@app.get("/health")
def health():
    return {
        "status": "healthy",
        "model_loaded": pipeline is not None,
        "n_classes": len(class_labels) if class_labels else 0,
    }


@app.get("/model-info")
def model_info():
    return {
        "model_type": metadata.get("model_type", "unknown"),
        "target_column": metadata.get("target_column", "crime_category"),
        "feature_columns": metadata.get("feature_columns", []),
        "class_labels": class_labels,
        "metrics": metadata.get("metrics", {}),
    }


@app.post("/predict", response_model=PredictionResponse)
def predict(payload: PredictionInput):
    row = {
        "hour": payload.hour,
        "day_of_week": payload.day_of_week,
        "month": payload.month,
        "is_weekend": payload.is_weekend,
        "community_area_id": payload.community_area_id,
        "distance_to_nearest_station": payload.distance_to_nearest_station,
        "stations_within_500m": payload.stations_within_500m,
        "community_type": payload.community_type,
    }

    X = pd.DataFrame([row])[feature_columns]
    probabilities = pipeline.predict_proba(X)[0]

    top_idx = probabilities.argsort()[::-1][:3]
    top_3_predictions = [
        {
            "label": class_labels[i],
            "probability": float(probabilities[i]),
        }
        for i in top_idx
    ]

    return PredictionResponse(
        predicted_label=top_3_predictions[0]["label"],
        predicted_probability=top_3_predictions[0]["probability"],
        top_3_predictions=top_3_predictions,
        model_version=metadata.get("model_type", "LightGBM multiclass pipeline"),
    )


/var/folders/cc/hct73g4d6mj23r23wb1283qw0000gn/T/ipykernel_28228/2477670297.py:97: DeprecationWarning: 
        on_event is deprecated, use lifespan event handlers instead.

        Read more about it in the
        [FastAPI docs for Lifespan Events](https://fastapi.tiangolo.com/advanced/events/).
        
  @app.on_event("startup")
